# Семинар 3. Продвинутый инжиниринг признаков (Feature Engineering)

**Цель семинара:** Научиться переводить сырые бизнес-метрики из статических профилей клиентов (`profiles.csv`) в математическую матрицу, пригодную для классического машинного обучения. Вы освоите создание возрастных корзин, расчет синтетических метрик, нормализацию и кодирование категорий. Итогом станет промышленная функция `engineer_profile_features` для пайплайна ядра.

### 🔧 Настройка окружения и импорт библиотек


In [ ]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings

from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)


---

## 📥 Шаг 1. Инициализация локального контура (Трек: Профили)

На этом семинаре мы продолжаем работать с мастер-таблицей `profiles.csv`, которую мы уже очистили на Семинаре 1. Задача — обогатить её новыми, сильными предикторами.


In [ ]:
BASE_DATA_DIR = os.path.join(".", "data")
INPUT_DIR = os.path.join(BASE_DATA_DIR, "seminar_2_rfm_clustering")
OUTPUT_DIR = os.path.join(BASE_DATA_DIR, "seminar_3_feature_engineering")
os.makedirs(OUTPUT_DIR, exist_ok=True)

profiles_clean_path = os.path.abspath(os.path.join(INPUT_DIR, "profiles_clean.csv"))

if not os.path.exists(profiles_clean_path):
    print(f"⚠️ Очищенный файл не найден. Генерируется mock-датасет для отработки Feature Engineering.")
    os.makedirs(INPUT_DIR, exist_ok=True)
    
    # Генерация синтетического чистого профиля
    mock_profiles = pd.DataFrame({
        'Target_ID': ['T01', 'T02', 'T03', 'T04', 'T05'],
        'Churn': [True, False, True, False, False],
        'tenure': [1, 15, 45, 70, 5],
        'MonthlyCharges': [50.0, 20.0, 100.0, 105.0, 45.5],
        'TotalCharges': [50.0, 300.0, 4500.0, 7350.0, 227.5],
        'gender': ['male', 'female', 'male', 'male', 'female'],
        'Contract': ['month-to-month', 'one_year', 'two_year', 'two_year', 'month-to-month'],
        'PaymentMethod': ['electronic_check', 'mailed_check', 'bank_transfer', 'credit_card', 'electronic_check']
    })
    mock_profiles.to_csv(profiles_clean_path, index=False)

print(f"Путь к чистым профилям инициализирован: {profiles_clean_path}")


---

## 🛠 ЗАДАНИЕ 1: Дискретизация числовых признаков (Binning)
**Бизнес-контекст:** Часто линейные алгоритмы ML плохо улавливают зависимости в непрерывных числах, но отлично работают с категориями. Разбив жизненный цикл клиента (`tenure`) на корзины лояльности (Новичок, Стабильный, Ветеран), мы поможем модели быстрее находить инсайты.

**Инструкция (TODO):**
1. Считайте датасет в `df_features`.
2. Используйте `pd.cut()` для разбиения колонки `tenure` на 3 категории лояльности (`Loyalty_Tier`).
   - Границы корзин (`bins`): `[-1, 12, 48, 100]` (до 1 года, 1-4 года, больше 4 лет).
   - Метки (`labels`): `['Newbie', 'Loyal', 'Veteran']`.

*🤖 Теги для AI-ментора: `#SEM3_TASK1_START`, `#SEM3_TASK1_BUG`*


In [ ]:
# [MASTER SOLUTION]
df_features = pd.read_csv(profiles_clean_path)

# Дискретизация через pd.cut
if 'tenure' in df_features.columns:
    df_features['Loyalty_Tier'] = pd.cut(
        df_features['tenure'], 
        bins=[-1, 12, 48, 100], 
        labels=['Newbie', 'Loyal', 'Veteran']
    )

print("Распределение по корзинам лояльности:")
display(df_features['Loyalty_Tier'].value_counts(dropna=False))


In [ ]:
# [STUDENT TEMPLATE]
# TODO: 1.1. Прочитайте CSV-файл profiles_clean_path в df_features
# TODO: 1.2. Проверьте наличие колонки 'tenure' в датафрейме
# TODO: 1.3. Примените функцию pd.cut() к колонке tenure. Передайте аргументы bins и labels.
# TODO: 1.4. Результат сохраните в новую колонку 'Loyalty_Tier'
raise NotImplementedError("Задание 1 не выполнено! Удалите эту строку и напишите свой код.")

df_features = pd.read_csv(...)

if 'tenure' in df_features.columns:
    df_features['Loyalty_Tier'] = pd.cut(
        x=..., 
        bins=[...], 
        labels=[...]
    )

display(df_features[['tenure', 'Loyalty_Tier']].head())


---

## 🛠 ЗАДАНИЕ 2: Инжиниринг синтетических бизнес-метрик (Ratios)
**Бизнес-контекст:** Сами по себе признаки могут быть слабыми предикторами, но их отношение — сильным. Например, отношение общих затрат к текущему платежу может указывать на скрытые наценки, а `Debt-to-Income` в банках — критический фактор дефолта.

**Инструкция (TODO):**
1. Создайте новый признак `Avg_Charge_Per_Month` (Средний исторический платеж), разделив `TotalCharges` на `tenure`.
2. Учтите ошибку деления на ноль! Если `tenure` равен 0, метрика должна быть равна 0. Используйте `np.where()`.

*🤖 Теги для AI-ментора: `#SEM3_TASK2_START`, `#SEM3_TASK2_BUG`*


In [ ]:
# [MASTER SOLUTION]
if all(col in df_features.columns for col in ['TotalCharges', 'tenure']):
    # Защита от деления на ноль через np.where
    df_features['Avg_Charge_Per_Month'] = np.where(
        df_features['tenure'] > 0,
        df_features['TotalCharges'] / df_features['tenure'],
        0.0
    )

display(df_features[['tenure', 'TotalCharges', 'Avg_Charge_Per_Month']].head())


In [ ]:
# [STUDENT TEMPLATE]
# TODO: 2.1. Проверьте, есть ли 'TotalCharges' и 'tenure' в колонках.
# TODO: 2.2. Используйте np.where(условие, значение_истина, значение_ложь). 
# Условие: tenure > 0. Истина: деление TotalCharges на tenure. Ложь: 0.0.
raise NotImplementedError("Задание 2 не выполнено! Удалите эту строку и напишите свой код.")

if ... and ... in df_features.columns:
    df_features['Avg_Charge_Per_Month'] = np.where(
        ..., # Условие
        ..., # Деление
        ...  # Ноль
    )


---

## 🛠 ЗАДАНИЕ 3: Кодирование текстовых категорий (One-Hot Encoding)
**Бизнес-контекст:** Модели машинного обучения (кроме некоторых реализаций деревьев) не понимают текст. Текстовые категории (`Contract`, `PaymentMethod`) необходимо перевести в бинарные матрицы нулей и единиц (0 и 1). Мы используем метод `pd.get_dummies()`.

**Инструкция (TODO):**
1. Сформируйте список колонок для кодирования: `['gender', 'Contract', 'PaymentMethod', 'Loyalty_Tier']`.
2. Примените метод `pd.get_dummies(df_features, columns=ваши_колонки, drop_first=True, dtype=float)`.
   - `drop_first=True` защищает от мультиколлинеарности (из двух полов оставляем один столбец, если не 1, значит 0).
3. Сохраните результат поверх `df_features`.

*🤖 Теги для AI-ментора: `#SEM3_TASK3_START`, `#SEM3_TASK3_WHY`*


In [ ]:
# [MASTER SOLUTION]
cat_to_encode = ['gender', 'Contract', 'PaymentMethod', 'Loyalty_Tier']

# Оставляем только те колонки, которые реально есть в датафрейме
cat_to_encode = [c for c in cat_to_encode if c in df_features.columns]

df_features = pd.get_dummies(
    df_features, 
    columns=cat_to_encode, 
    drop_first=True, 
    dtype=float # Чтобы получить 1.0 и 0.0, а не True/False
)

print(f"Размерность после One-Hot Encoding: {df_features.shape}")
display(df_features.head(3))


In [ ]:
# [STUDENT TEMPLATE]
# TODO: 3.1. Создайте список категориальных колонок (например, gender, Contract, PaymentMethod, Loyalty_Tier)
# TODO: 3.2. Вызовите pd.get_dummies(), передав туда датафрейм, список колонок, drop_first=True и dtype=float
raise NotImplementedError("Задание 3 не выполнено! Удалите эту строку и напишите свой код.")

cat_to_encode = [...]
cat_to_encode = [c for c in cat_to_encode if c in df_features.columns]

df_features = pd.get_dummies(
    ..., 
    columns=..., 
    drop_first=..., 
    dtype=...
)


---

## 🛠 ЗАДАНИЕ 4: Масштабирование признаков (StandardScaler)
**Бизнес-контекст:** Признак `TotalCharges` измеряется тысячами, а `tenure` — десятками. Линейные модели и нейросети будут придавать больший вес признакам с широким диапазоном. Мы нормализуем метрические данные, чтобы среднее стало 0, а стандартное отклонение 1.

**Инструкция (TODO):**
1. Выделите числовые непрерывные колонки: `['tenure', 'MonthlyCharges', 'TotalCharges', 'Avg_Charge_Per_Month']`.
2. Инициализируйте `StandardScaler()`.
3. Обучите и преобразуйте выбранные колонки с помощью `.fit_transform()`. Сохраните результат в те же колонки.

*🤖 Теги для AI-ментора: `#SEM3_TASK4_START`, `#SEM3_TASK4_BUG`*


In [ ]:
# [MASTER SOLUTION]
numeric_to_scale = ['tenure', 'MonthlyCharges', 'TotalCharges', 'Avg_Charge_Per_Month']
numeric_to_scale = [c for c in numeric_to_scale if c in df_features.columns]

scaler = StandardScaler()

if numeric_to_scale:
    # Важно: StandardScaler возвращает numpy array, поэтому записываем обратно в df
    df_features[numeric_to_scale] = scaler.fit_transform(df_features[numeric_to_scale])

print("Признаки отмасштабированы. Средние значения стремятся к 0:")
display(df_features[numeric_to_scale].mean().round(3))


In [ ]:
# [STUDENT TEMPLATE]
# TODO: 4.1. Определите список числовых колонок для нормализации
# TODO: 4.2. Инициализируйте объект StandardScaler()
# TODO: 4.3. Примените метод .fit_transform() к этим колонкам и перезапишите их в df_features
raise NotImplementedError("Задание 4 не выполнено! Удалите эту строку и напишите свой код.")

numeric_to_scale = [...]
numeric_to_scale = [c for c in numeric_to_scale if c in df_features.columns]

scaler = ...

if numeric_to_scale:
    df_features[...] = scaler....(df_features[...])


---

## 🛠 ЗАДАНИЕ 5: Аудит мультиколлинеарности
**Бизнес-контекст:** Сильно коррелированные признаки (коэффициент Пирсона > 0.85) дублируют информацию и дестабилизируют веса линейных алгоритмов машинного обучения. От них нужно избавляться.

**Инструкция (TODO):**
1. Оставьте в датафрейме только числовые колонки с помощью `.select_dtypes(include=[np.number])`.
2. Постройте матрицу корреляций с помощью `.corr()`.
3. Визуализируйте матрицу через `sns.heatmap()`. Наша задача — проанализировать тепловую карту глазками и обосновать удаление колонок.

*🤖 Теги для AI-ментора: `#SEM3_TASK5_WHY`*


In [ ]:
# [MASTER SOLUTION]
# Берем только числа
numeric_df = df_features.select_dtypes(include=[np.number, float, int])

# Расчет матрицы корреляции Пирсона
corr_matrix = numeric_df.corr()

# Визуализация
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title('Матрица корреляции Пирсона')
plt.show()

# Анализ: Видно, что TotalCharges имеет сильную линейную связь (мультиколлинеарность). 
# В production мы бы могли дропнуть её: df_features.drop(columns=['TotalCharges'], inplace=True)


In [ ]:
# [STUDENT TEMPLATE]
# TODO: 5.1. Отфильтруйте df_features, оставив только числовые типы
# TODO: 5.2. Вычислите .corr()
# TODO: 5.3. Постройте sns.heatmap(corr_matrix, annot=True, cmap='coolwarm')
raise NotImplementedError("Задание 5 не выполнено! Удалите эту строку и напишите свой код.")

numeric_df = df_features.select_dtypes(...)
corr_matrix = numeric_df...

plt.figure(figsize=(10, 8))
sns...
plt.title('Матрица корреляции Пирсона')
plt.show()


---

## 🏗 ФИНАЛЬНАЯ СБОРКА: Сквозная функция engineer_profile_features

Объединяем наш инжиниринг в единую production-функцию. Именно она будет вызываться при обработке новых клиентов на сервере FastAPI.


In [ ]:
# [MASTER SOLUTION]
def engineer_profile_features(
    df: pd.DataFrame, 
    categorical_to_encode: list = None,
    numeric_to_scale: list = None
) -> pd.DataFrame:
    """
    Промышленная функция генерации и кодирования признаков профиля сущности.
    Включает расчет метрик, OHE-кодирование и масштабирование.
    """
    df_engineered = df.copy()
    
    # 1. Синтетические метрики
    if all(col in df_engineered.columns for col in ['TotalCharges', 'tenure']):
        df_engineered['Avg_Charge_Per_Month'] = np.where(
            df_engineered['tenure'] > 0,
            df_engineered['TotalCharges'] / df_engineered['tenure'],
            0.0
        )
        
    # 2. Дискретизация (Hardcoded для домена)
    if 'tenure' in df_engineered.columns:
        df_engineered['Loyalty_Tier'] = pd.cut(
            df_engineered['tenure'], 
            bins=[-1, 12, 48, 100], 
            labels=['Newbie', 'Loyal', 'Veteran']
        )
        # Добавляем новую колонку в список кодирования, если её там нет
        if categorical_to_encode and 'Loyalty_Tier' not in categorical_to_encode:
            categorical_to_encode.append('Loyalty_Tier')

    # 3. One-Hot Encoding
    if categorical_to_encode:
        actual_cat = [c for c in categorical_to_encode if c in df_engineered.columns]
        if actual_cat:
            df_engineered = pd.get_dummies(df_engineered, columns=actual_cat, drop_first=True, dtype=float)

    # 4. Scaling
    if numeric_to_scale:
        actual_num = [c for c in numeric_to_scale if c in df_engineered.columns]
        if actual_num:
            scaler = StandardScaler()
            df_engineered[actual_num] = scaler.fit_transform(df_engineered[actual_num])

    return df_engineered

# Применение пайплайна
df_final = engineer_profile_features(
    pd.read_csv(profiles_clean_path),
    categorical_to_encode=['gender', 'Contract', 'PaymentMethod'],
    numeric_to_scale=['tenure', 'MonthlyCharges', 'TotalCharges']
)
print(f"Feature Engineering завершен. Размерность: {df_final.shape}")


In [ ]:
# [STUDENT TEMPLATE]
def engineer_profile_features(
    df: pd.DataFrame, 
    categorical_to_encode: list = None,
    numeric_to_scale: list = None
) -> pd.DataFrame:
    """
    Промышленная параметризованная функция Feature Engineering.
    """
    # TODO: Соберите архитектуру функции, опираясь на параметры categorical_to_encode и numeric_to_scale
    raise NotImplementedError("Финальная сборка функции не выполнена!")
    
    df_engineered = df.copy()
    
    # 1. Синтетические метрики
    # ...
    
    # 2. Дискретизация
    # ...
    
    # 3. One-Hot Encoding
    # ...
    
    # 4. Scaling
    # ...
    
    return df_engineered

df_final = engineer_profile_features(pd.read_csv(profiles_clean_path), [], [])


---

## 🛠 Автоматизированная проверка качества (Autocheck)

Скрипт проверяет, правильно ли отработали методы One-Hot Encoding и StandardScaler.


In [ ]:
def run_autocheck(df_to_check):
    print("🚀 Тестирование функции engineer_profile_features...\n" + "-"*45)
    validation_status = True
    
    if not isinstance(df_to_check, pd.DataFrame):
        print("❌ Ошибка: Результирующий объект не является DataFrame.")
        return False
        
    # Проверка OHE (отсутствие типа Object)
    object_cols = df_to_check.select_dtypes(include=['object']).columns
    # Исключаем Target_ID из проверки, так как это ключ
    object_cols = [c for c in object_cols if c != 'Target_ID']
    
    if len(object_cols) > 0:
        print(f"❌ Ошибка: В таблице остались некодированные строковые категории: {object_cols}")
        validation_status = False
    else:
        print("✅ Все категориальные признаки успешно переведены в числовые флаги (OHE).")
        
    # Проверка масштабирования (Std ~ 1.0)
    if 'tenure' in df_to_check.columns:
        std_val = df_to_check['tenure'].std()
        if not (0.9 < std_val < 1.2):
            print(f"❌ Ошибка: Стандартное отклонение числовых фичей не равно ~1.0 (Текущее: {std_val:.2f}). Scaling не выполнен.")
            validation_status = False
        else:
            print("✅ Масштабирование (StandardScaler) применено корректно.")
            
    # Проверка синтетической фичи
    if 'Avg_Charge_Per_Month' in df_to_check.columns:
        print("✅ Синтетический бизнес-показатель успешно добавлен.")
    else:
        print("❌ Ошибка: Синтетическая колонка Avg_Charge_Per_Month не найдена.")
        validation_status = False

    print("-" * 45)
    if validation_status:
        print("🎉 ПОЗДРАВЛЯЕМ! Пайплайн подготовки профилей соответствует стандартам.")
        print("Перенесите эту функцию в course_project/src/data_pipeline.py!")
    else:
        print("⚠️ Обнаружены технические дефекты. Проверьте реализацию функции.")

run_autocheck(df_final)
